# Libraries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import optuna
from pathlib import Path

# -- Personal Libraries
from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, MultiProductContextEmbeddings
from src.utils import TemporalSplitter

/home/thebigmonster/Github/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setting

In [3]:
# initial seed
BASE_SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Data ──────────────────────────────────────────────────────────
N_UPCS = 5 # Number of UPCs
SMOOTH_WINDOW = 8 # Smoothing window for phase 0
BETA_EDA = -2 # Beta for initialization phase 0

# ── Robust Tuning ─────────────────────────────────────────────────
N_FOLDS = 3  # Number of folds for cross-validation
TUNE_SEEDS = [11, 29, 42]  # Seeds for cross-validation
MIN_TRAIN_FRAC = 0.50  # Minimum training fraction

# ── Training for tuning ──────────────────────────────────────
N_EPOCHS_P0 = 200
N_EPOCHS_P1 = 200
N_EPOCHS_P2 = 250
PATIENCE    = 20 # How many epochs to wait before reducing learning rate
ES_PATIENCE = 40 # How many epochs to wait before early stopping

# ── Checkpoints ────────────────────────────────────────────────────
CKPT_DIR = Path("../results/checkpoints/hparam")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Results ─────────────────────────────────────────────────────
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_TRIAL_PATH = RESULTS_DIR / "best_trial_params.json"
TRIAL_SUMMARY_PATH = RESULTS_DIR / "nn_hparam_trials_summary.csv"

Device: cuda


In [4]:
# Function to set all seeds
# and make the results reproducible
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Make the results reproducible and control the randomness
    torch.backends.cudnn.benchmark = False # Make the results reproducible and control the randomness

set_all_seeds(BASE_SEED)

In [ ]:
# Load the dataset
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()
print(f"Dataset shape: {df.shape}")

# Encode the categorical variables
encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True) # Encode the store code to numerical values
_, week_cats  = encoder.factorize(df, "week_id", sort=True) # Encode the week id to numerical values

n_stores = len(store_cats)
n_weeks  = len(week_cats)
print(f"Stores: {n_stores}  |  Weeks: {n_weeks}")

# Build the multi-product dataset
mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS) # Fit the builder to the data

# Transform the data to wide format (pivot table with UPCs and regressors as a columns
# and week_store as rows)
full_wide_raw = mp_builder.transform().copy() 
n_upcs = mp_builder.n # Store the number of selected UPCs

print(f"Full wide shape: {full_wide_raw.shape}")
print(f"UPCs selected: {n_upcs}")
print(f"Top UPCs: {mp_builder.selected_upcs[:N_UPCS]}")

Dataset shape: (463722, 30)
Stores: 70  |  Weeks: 302
Full wide shape: (19808, 101)
UPCs seleccionados: 5
Top UPCs: [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]


In [ ]:
splitter = TemporalSplitter(week_col="week_id") # Initialize the temporal splitter
fold_splits = splitter.expanding_splits(
    df=full_wide_raw, # The data to split
    n_folds=N_FOLDS, # The number of folds
    min_train_frac=MIN_TRAIN_FRAC, # The minimum training fraction
)

print(f"N folds available: {len(fold_splits)}")
for i, (train_fold, val_fold) in enumerate(fold_splits):
    print(
        f"Fold {i}: train={len(train_fold):,} "
        f"val={len(val_fold):,} "
        f"train_weeks={train_fold['week_id'].nunique()} "
        f"val_weeks={val_fold['week_id'].nunique()}"
    )

N folds disponibles: 3
Fold 0: train=9,756 val=3,394 train_weeks=151 val_weeks=50
Fold 1: train=13,150 val=3,339 train_weeks=201 val_weeks=50
Fold 2: train=16,489 val=3,254 train_weeks=251 val_weeks=50


In [ ]:
# We create a mapping of store and week (numerical)codes to 0,1,2,...
# to be globally used for the folds; For instance,
# store_cats = Index([101, 102,...])
# store_map = {101: 0, 102: 1, ...}
# The same for week_cats and week_map.
store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}


# This function prepare the data for training.
# It encodes the store and week codes, sorts the data by store and week codes,
# and smooths the log liters.
def build_fold_frames(train_wide, val_wide, smooth_window: int):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    # Encode the store and week codes
    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    # Sort the data by store and week codes to do the rolling mean
    train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
    val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()

    # Smooth the log liters. Delete the noise week by week.
    # For the Phase 0, we use a moving average of n weeks.
    for i in range(n_upcs):
        col = f"log_liters_{i}"
        for df_w in [train_wide_s, val_wide_s]:
            df_w[col] = (
                df_w.groupby("store_code")[col]
                .transform(lambda s: s.rolling(window=smooth_window, min_periods=1).mean())
            )

    return train_wide, val_wide, train_wide_s, val_wide_s

# This function builds the datasets for the training and validation.
def build_fold_datasets(train_wide, val_wide, train_wide_s, val_wide_s):
    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs) # Phase 0 training dataset
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs) # Phase 0 validation dataset
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs) # Phase 1/2 training dataset
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs) # Phase 1/2 validation dataset
    return train_ds_p0, val_ds_p0, train_ds, val_ds

In [ ]:
# This function runs the training loop.
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, phase_name="", verbose=False):

    best_val_loss = float("inf") # Initialize the best validation loss
    no_improve    = 0 # Initialize the number of epochs without improvement
    # Scales the loss to prevent underflow in training with mixed precision (float32->float16)
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    # Training loop
    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train() # Set the model to training mode
        total_loss, total_denom = 0.0, 0.0 # Initialize the total loss and the pondered denominator
        # The batches don't have the same size, because it exists the obs_mask (observations mask);
        # we can't treat a batch with 10 observation like one with 100 observations. For this reason, 
        # we need to get the pondered real average.

        # Recall that: obs_mask = 1 if the observation is available
        # (the product was sold this week in this store), 0 otherwise (the product was not sold).

        for batch in train_loader:
            batch    = {k: v.to(device) for k, v in batch.items()}
            # It arrives a batch as:
            # batch = {
            #     "log_liters_0": tensor([...]),  # shape: (batch_size,)
            #     "log_liters_1": tensor([...]),  # shape: (batch_size,)
            #     "log_liters_2": tensor([...]),  # shape: (batch_size,)
            #     ...
            # }
            # therefore, we need to stack the log_liters and obs_mask for each product
            # in the batch.
            # Important! Why are these variables stacked and not just inside the model?
            # Because these variables are not used to compute the prediction,
            # they are only used to compute the loss.
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

            optimizer.zero_grad() # Reset the gradients
            if scaler: # If the scaler is not None, we use mixed precision
                with torch.amp.autocast("cuda"): # Use mixed precision (AMP)
                    y_hat, eps_hat, aux = model(batch, return_parts=True) # Get the prediction
                    loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                        aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                        aux["Bx"], aux["IBx"],
                                        model.head.param_head._pairs) # Compute the loss
                scaler.scale(loss).backward() # Backward pass
                # Unscale the gradients; the gradients are inflated 
                # because of the mixed precision (float32->float16).
                scaler.unscale_(optimizer)
                # We need to avoid explosive gradients, for this reason
                # we clip the gradients
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer) # We update the parameters
                scaler.update() # We update the scale factor of the scaler
            else:
                # If the scaler is None (no GPU), we don't use mixed precision
                # and we use the normal backward pass.
                y_hat, eps_hat, aux = model(batch, return_parts=True)
                loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                    aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                    aux["Bx"], aux["IBx"],
                                    model.head.param_head._pairs)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            denom        = obs_mask.sum().item() # Number of available observations (n_obs_batch)
            # Recall that:
            # logs["loss"] = total_loss_batch / n_obs_batch
            # We recover the total loss to, at the end of the epoch,
            # compute the real average.
            total_loss  += logs["loss"].item() * denom 
            total_denom += denom # Sum of the denominator

        # ── Val ────────────────────────────────────────────────────
        model.eval() # Set the model to evaluation mode
        val_loss_sum, val_denom = 0.0, 0.0 # Initialize the validation loss and the pondered denominator

        with torch.no_grad(): # No gradients are computed
            for batch in val_loader: # Iterate over the validation loader
                batch    = {k: v.to(device) for k, v in batch.items()} # Move the batch to the device
                y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1) # Stack the log_liters
                obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1) # Stack the obs_mask

                y_hat, eps_hat, aux = model(batch, return_parts=True) # Get the prediction
                _, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                  aux["w"], aux["ddBx"], aux["dddBx"], aux["u"],
                                  aux["Bx"], aux["IBx"],
                                  model.head.param_head._pairs) # Compute the loss
                                  
                denom        = obs_mask.sum().item() # Number of available observations
                val_loss_sum += logs["loss"].item() * denom # Sum of the total loss
                val_denom    += denom # Sum of the denominator

        # We compute the pondered real average.
        val_loss = val_loss_sum / max(val_denom, 1.0) # Average of the loss
        prev_lr = optimizer.param_groups[0]["lr"] # Previous learning rate
        scheduler.step(val_loss) # Update the learning rate (scheduler)
        new_lr = optimizer.param_groups[0]["lr"] # New learning rate
        if new_lr < prev_lr: # If the new learning rate is lower than the previous one,
            no_improve = 0

        # If the validation loss is lower than the best validation loss,
        # we save the model otherwise we increment the number of epochs without improvement.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        # If the number of epochs without improvement is 0,
        # we print the validation loss.
        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")

        # If the number of epochs without improvement is greater than the patience,
        # we stop the training (Early Stopping).
        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping en época {epoch+1}")
            break

    return best_val_loss

print("run_training defined")

run_training definida


In [ ]:
# Hidden options for the model (Optuna)
HIDDEN_OPTIONS = {
    "64_32":      (64, 32),
    "128_64_32":  (128, 64, 32),
    "64_32_16":   (64, 32, 16),
    "128_64":     (128, 64),
}

# This function compute the R2, MAE and RMSE.
# Recall that:
# MAE is the mean absolute error.
# RMSE is the root mean square error.
# R2 is the coefficient of determination.
def compute_global_metrics(model, val_loader, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device) for k, v in batch.items()}
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)
            y_hat, _, _ = model(batch, return_parts=True)

            # We get only the available observations.
            mask = obs_mask.bool() # Mask of the available observations
            all_true.append(y_true[mask].cpu()) # Append the true values
            all_pred.append(y_hat[mask].cpu()) # Append the predicted values

    y_true_all = torch.cat(all_true).float() # Concatenate the true values
    y_pred_all = torch.cat(all_pred).float() # Concatenate the predicted values

    err = y_true_all - y_pred_all # Error
    mae = float(err.abs().mean()) # Mean absolute error
    rmse = float(torch.sqrt((err ** 2).mean())) # Root mean square error

    ss_res = float((err ** 2).sum()) # Sum of the squared errors
    ss_tot = float(((y_true_all - y_true_all.mean()) ** 2).sum()) # Sum of the total errors
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan # R2

    return {
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2,
    }

# This function compute the Elasticity Score for the optimization parameters of Optuna.
# Our intention is to evaluate how good the model is at predicting the elasticity.
# Recall that:
# The Elasticity Score is in the range [0, 1].
# The closer to 1, the better.
# We shall assume that in FMCG, tipically the elasticity is in the range [-5, 0]. 
# One could change this range to adapt it to other products, but it is not the purpose of this notebook.
def compute_elasticity_score(model, val_loader, device, elast_min=-5.0, elast_max=0.0):
    model.eval()
    all_elast = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            obs_mask = torch.stack([batch[f"obs_mask_{i}"] for i in range(model.n)], dim=1).bool()
            _, eps_hat, _ = model(batch, return_parts=True)
            # Append the predicted elasticities only on the available observations
            all_elast.append(eps_hat[obs_mask].cpu()) 

    elast = torch.cat(all_elast).numpy() # Concatenate the predicted elasticities

    # We compute the percentage of predicted elasticities that are in the range [-5, 0].
    in_range = float(((elast >= elast_min) & (elast <= elast_max)).mean())
    median_e = float(np.median(elast)) # Median of the predicted elasticities

    # We compute the penalty for the prior.
    # Because of EDA, the global elasticity is -2 approximately.
    # Therefore, we want the median of the predicted elasticities to be -2.
    # If it is not, we penalize the model.
    deviation = max(0.0, abs(median_e - BETA_EDA) - 0.3)
    prior_penalty = min(deviation / abs(BETA_EDA), 1.0)

    # We compute the final score.
    # It is a weighted average of the percentage of predicted elasticities in the range [-5, 0]
    # and the penalty for the prior.
    score = in_range * (1.0 - prior_penalty)

    return {
        "elast_score": float(score),
        "elasticity_median": median_e,
        "elasticity_in_range": float(in_range),
    }

print("Helpers of metrics defined")

Helpers de métricas definidos


In [24]:
def build_and_train(params, train_fold, val_fold, fold_id, seed, trial_id=0):
    set_all_seeds(seed)

    train_wide, val_wide, train_wide_s, val_wide_s = build_fold_frames(
        train_wide=train_fold,
        val_wide=val_fold,
        smooth_window=SMOOTH_WINDOW,
    )

    train_ds_p0, val_ds_p0, train_ds, val_ds = build_fold_datasets(
        train_wide, val_wide, train_wide_s, val_wide_s
    )

    n_knots          = params["N_KNOTS"]
    hidden           = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
    dropout          = params["DROPOUT"]
    d_store          = params.get("D_STORE", 16)
    act              = params.get("ACT", "gelu")
    lr_p0            = params["LR_P0"]
    lr_p1            = params["LR_P1"]
    lr_p2            = params["LR_P2"]
    lambda_smooth_p2 = params["LAMBDA_SMOOTH_P2"]
    lambda_pos_p2    = params["LAMBDA_POS_P2"]
    batch_size       = params["BATCH_SIZE"]

    ckpt_p0 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase1.pt"
    ckpt_p2 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase2.pt"

    loader_factory = DataLoaderFactory(num_workers=0, pin_memory=True)
    train_loader_p0 = loader_factory.create_train_loader(
        train_ds_p0, batch_size=batch_size, shuffle=True, drop_last=True
    )
    val_loader_p0 = loader_factory.create_eval_loader(
        val_ds_p0, batch_size=batch_size, shuffle=False
    )
    train_loader = loader_factory.create_train_loader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True
    )
    val_loader = loader_factory.create_eval_loader(
        val_ds, batch_size=batch_size, shuffle=False
    )

    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        x_i = train_wide[f"log_price_{i}"].values
        config = builder.build_from_data(x_i, n_knots=n_knots, q_min=0.05, q_max=0.95)
        spline_configs.append(config)

    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"]  for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]   for cfg in spline_configs])
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    cb = MultiProductContextEmbeddings(
        n=n_upcs,
        n_stores=n_stores,
        d_store=d_store,
    )

    def make_model(enforce_negative_beta, use_cross):
        head = IntegrableDemandHead(
            context_dim=cb.out_dim,
            K_splines=n_knots,
            n=n_upcs,
            hidden=hidden,
            act=act,
            dropout=dropout,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        return ICDN(
            context_builder=cb,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)

    # ── FASE 0 ─────────────────────────────────────────────────────
    m0 = make_model(enforce_negative_beta=True, use_cross=False)
    with torch.no_grad():
        m0.head.param_head.head_w.weight.zero_()
        m0.head.param_head.head_w.bias.zero_()
    m0.head.param_head.head_w.weight.requires_grad_(False)
    m0.head.param_head.head_w.bias.requires_grad_(False)

    beta_raw_init = torch.log(torch.exp(torch.tensor(-BETA_EDA, dtype=torch.float32)) - 1.0)
    with torch.no_grad():
        m0.head.param_head.head_beta.weight.zero_()
        m0.head.param_head.head_beta.bias.fill_(beta_raw_init)

    loss_p0 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    decay, no_decay = [], []
    for name, p in m0.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p0,
    )
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m0, train_loader_p0, val_loader_p0, loss_p0,
                 opt_p0, sch_p0, N_EPOCHS_P0, ES_PATIENCE, ckpt_p0, device, "P0")

    # ── FASE 1 ─────────────────────────────────────────────────────
    m1 = make_model(enforce_negative_beta=True, use_cross=False)
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    m1.head.param_head.head_w.weight.requires_grad_(True)
    m1.head.param_head.head_w.bias.requires_grad_(True)

    loss_p1 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    decay, no_decay = [], []
    for name, p in m1.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p1,
    )
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m1, train_loader, val_loader, loss_p1,
                 opt_p1, sch_p1, N_EPOCHS_P1, ES_PATIENCE, ckpt_p1, device, "P1")

    # ── FASE 2 ─────────────────────────────────────────────────────
    m2 = make_model(enforce_negative_beta=True, use_cross=True)
    state = torch.load(ckpt_p1, map_location=device)
    state.pop("head.param_head._pairs", None)
    m2.load_state_dict(state, strict=False)

    m2.head.param_head.head_w.weight.requires_grad_(True)
    m2.head.param_head.head_w.bias.requires_grad_(True)
    with torch.no_grad():
        m2.head.param_head.head_cross.weight.zero_()
        m2.head.param_head.head_cross.bias.zero_()

    loss_p2 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=lambda_smooth_p2,
        lambda_pos=lambda_pos_p2,
        reduction="none",
    )
    decay, no_decay = [], []
    for name, p in m2.named_parameters():
        if not p.requires_grad:
            continue
        if ("head_w" in name) or ("head_cross" in name) or name.endswith("bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    opt_p2 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p2,
    )
    sch_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p2, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )
    run_training(m2, train_loader, val_loader, loss_p2,
                 opt_p2, sch_p2, N_EPOCHS_P2, ES_PATIENCE, ckpt_p2, device, "P2")

    m2.load_state_dict(torch.load(ckpt_p2, map_location=device))

    pred_metrics = compute_global_metrics(m2, val_loader, device)
    elast_metrics = compute_elasticity_score(m2, val_loader, device)

    out = {
        "trial_id": trial_id,
        "fold": fold_id,
        "seed": seed,
        "n_train": len(train_wide),
        "n_val": len(val_wide),
        **pred_metrics,
        **elast_metrics,
    }

    print(
        f"trial={trial_id} fold={fold_id} seed={seed} | "
        f"R2={out['r2_val']:.4f} MAE={out['mae_val']:.4f} "
        f"ElastScore={out['elast_score']:.4f}"
    )

    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)
    ckpt_p2.unlink(missing_ok=True)

    return out

print("build_and_train redefinida")

build_and_train redefinida


In [25]:
trial_records = []

def objective(trial):
    params = {
        "N_KNOTS":          trial.suggest_int("N_KNOTS", 2, 16),
        "HIDDEN_KEY":       trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS.keys())),
        "DROPOUT":          trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR_P0":            trial.suggest_float("LR_P0",  1e-4, 1e-2, log=True),
        "LR_P1":            trial.suggest_float("LR_P1",  1e-5, 5e-3, log=True),
        "LR_P2":            trial.suggest_float("LR_P2",  1e-5, 1e-3, log=True),
        "LAMBDA_SMOOTH_P2": trial.suggest_float("LAMBDA_SMOOTH_P2", 1e-5, 0.2, log=True),
        "LAMBDA_POS_P2":    trial.suggest_float("LAMBDA_POS_P2",    0.05, 0.5),
        "BATCH_SIZE":       trial.suggest_categorical("BATCH_SIZE", [16, 32, 64]),
    }

    print(f"\n{'='*70}")
    print(f"Trial {trial.number}")
    for k, v in params.items():
        print(f"  {k}: {v}")
    print(f"{'='*70}")

    run_rows = []
    for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
        for seed in TUNE_SEEDS:
            row = build_and_train(
                params=params,
                train_fold=train_fold,
                val_fold=val_fold,
                fold_id=fold_id,
                seed=seed,
                trial_id=trial.number,
            )
            run_rows.append(row)

    df_trial = pd.DataFrame(run_rows)

    mean_r2 = float(df_trial["r2_val"].mean())
    std_r2  = float(df_trial["r2_val"].std(ddof=1)) if len(df_trial) > 1 else 0.0

    mean_elast = float(df_trial["elast_score"].mean())
    std_elast  = float(df_trial["elast_score"].std(ddof=1)) if len(df_trial) > 1 else 0.0

    mean_mae  = float(df_trial["mae_val"].mean())
    mean_rmse = float(df_trial["rmse_val"].mean())

    robust_r2 = mean_r2 - 0.25 * std_r2
    robust_elast = mean_elast - 0.25 * std_elast

    trial.set_user_attr("mean_r2", mean_r2)
    trial.set_user_attr("std_r2", std_r2)
    trial.set_user_attr("mean_elast_score", mean_elast)
    trial.set_user_attr("std_elast_score", std_elast)
    trial.set_user_attr("mean_mae", mean_mae)
    trial.set_user_attr("mean_rmse", mean_rmse)
    trial.set_user_attr("robust_r2", robust_r2)
    trial.set_user_attr("robust_elast", robust_elast)

    df_trial["trial"] = trial.number
    for k, v in params.items():
        df_trial[k] = v
    trial_records.extend(df_trial.to_dict(orient="records"))

    print(
        f"Trial {trial.number} summary | "
        f"mean_R2={mean_r2:.4f} std_R2={std_r2:.4f} "
        f"robust_R2={robust_r2:.4f} | "
        f"mean_Elast={mean_elast:.4f} std_Elast={std_elast:.4f} "
        f"robust_Elast={robust_elast:.4f}"
    )

    return robust_r2, robust_elast

In [26]:
study = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="hparam_pareto_kfold_seed",
    storage="sqlite:///../results/hparam_pareto_kfold_seed.db",
    load_if_exists=True,
)

study.optimize(objective, n_trials=10)

print(f"\nTrials completados: {len(study.trials)}")
print(f"Trials Pareto-óptimos: {len(study.best_trials)}")

[I 2026-03-23 19:43:52,819] Using an existing study with name 'hparam_pareto_kfold_seed' instead of creating a new one.



Trial 44
  N_KNOTS: 9
  HIDDEN_KEY: 128_64_32
  DROPOUT: 0.16669272015879266
  LR_P0: 0.0038242644355846044
  LR_P1: 0.002693117910194053
  LR_P2: 2.143194332811135e-05
  LAMBDA_SMOOTH_P2: 0.001399261434897975
  LAMBDA_POS_P2: 0.3309022623310561
  BATCH_SIZE: 32
trial=44 fold=0 seed=11 | R2=0.6631 MAE=0.5804 ElastScore=0.9764
trial=44 fold=0 seed=29 | R2=0.6951 MAE=0.5480 ElastScore=0.9626
trial=44 fold=0 seed=42 | R2=0.6430 MAE=0.5965 ElastScore=0.9768
trial=44 fold=1 seed=11 | R2=-5.3607 MAE=0.6566 ElastScore=0.7754
trial=44 fold=1 seed=29 | R2=0.4853 MAE=0.6338 ElastScore=0.9715
trial=44 fold=1 seed=42 | R2=0.6284 MAE=0.5258 ElastScore=0.9956
trial=44 fold=2 seed=11 | R2=0.3000 MAE=0.5727 ElastScore=1.0000
trial=44 fold=2 seed=29 | R2=0.3610 MAE=0.5460 ElastScore=1.0000


[I 2026-03-24 00:52:29,513] Trial 44 finished with values: [-0.6300328167936311, 0.9441759694881161] and parameters: {'N_KNOTS': 9, 'HIDDEN_KEY': '128_64_32', 'DROPOUT': 0.16669272015879266, 'LR_P0': 0.0038242644355846044, 'LR_P1': 0.002693117910194053, 'LR_P2': 2.143194332811135e-05, 'LAMBDA_SMOOTH_P2': 0.001399261434897975, 'LAMBDA_POS_P2': 0.3309022623310561, 'BATCH_SIZE': 32}.


trial=44 fold=2 seed=42 | R2=0.3337 MAE=0.5603 ElastScore=1.0000
Trial 44 summary | mean_R2=-0.1390 std_R2=1.9641 robust_R2=-0.6300 | mean_Elast=0.9620 std_Elast=0.0714 robust_Elast=0.9442

Trial 45
  N_KNOTS: 5
  HIDDEN_KEY: 128_64_32
  DROPOUT: 0.14729256303492735
  LR_P0: 0.007878366622352582
  LR_P1: 0.0001240188761204096
  LR_P2: 6.41791149160687e-05
  LAMBDA_SMOOTH_P2: 0.000953808392018296
  LAMBDA_POS_P2: 0.443029115754364
  BATCH_SIZE: 32
trial=45 fold=0 seed=11 | R2=0.6784 MAE=0.5676 ElastScore=0.3437
trial=45 fold=0 seed=29 | R2=0.6814 MAE=0.5641 ElastScore=0.3653
trial=45 fold=0 seed=42 | R2=0.6857 MAE=0.5595 ElastScore=0.3873
trial=45 fold=1 seed=11 | R2=0.5576 MAE=0.5802 ElastScore=0.3251
trial=45 fold=1 seed=29 | R2=0.5778 MAE=0.5631 ElastScore=0.2661
trial=45 fold=1 seed=42 | R2=0.5909 MAE=0.5484 ElastScore=0.2779
trial=45 fold=2 seed=11 | R2=0.3824 MAE=0.5387 ElastScore=0.8004
trial=45 fold=2 seed=29 | R2=0.3916 MAE=0.5378 ElastScore=0.6418


[I 2026-03-24 04:58:02,145] Trial 45 finished with values: [0.521360119505604, 0.41940639301669286] and parameters: {'N_KNOTS': 5, 'HIDDEN_KEY': '128_64_32', 'DROPOUT': 0.14729256303492735, 'LR_P0': 0.007878366622352582, 'LR_P1': 0.0001240188761204096, 'LR_P2': 6.41791149160687e-05, 'LAMBDA_SMOOTH_P2': 0.000953808392018296, 'LAMBDA_POS_P2': 0.443029115754364, 'BATCH_SIZE': 32}.


...

trial=51 fold=2 seed=42 | R2=0.4634 MAE=0.4980 ElastScore=0.9079
Trial 51 summary | mean_R2=0.5612 std_R2=0.1112 robust_R2=0.5334 | mean_Elast=0.8650 std_Elast=0.1585 robust_Elast=0.8253

Trial 52
  N_KNOTS: 9
  HIDDEN_KEY: 128_64
  DROPOUT: 0.09943242510363191
  LR_P0: 0.0005371582324154472
  LR_P1: 0.00011047117881763945
  LR_P2: 0.0002965926016432513
  LAMBDA_SMOOTH_P2: 0.00012227781489711028
  LAMBDA_POS_P2: 0.4864764033332494
  BATCH_SIZE: 32
trial=52 fold=0 seed=11 | R2=0.7246 MAE=0.5158 ElastScore=0.3460
trial=52 fold=0 seed=29 | R2=0.7074 MAE=0.5367 ElastScore=0.3337
trial=52 fold=0 seed=42 | R2=0.7029 MAE=0.5349 ElastScore=0.3819
trial=52 fold=1 seed=11 | R2=0.6413 MAE=0.5229 ElastScore=0.5350
trial=52 fold=1 seed=29 | R2=0.5717 MAE=0.5436 ElastScore=0.5585
trial=52 fold=1 seed=42 | R2=0.4702 MAE=0.5482 ElastScore=0.6537
trial=52 fold=2 seed=11 | R2=0.5123 MAE=0.4809 ElastScore=0.8945
trial=52 fold=2 seed=29 | R2=0.5241 MAE=0.4728 ElastScore=0.8200


[I 2026-03-25 16:38:28,145] Trial 52 finished with values: [0.5709306843602959, 0.54550983006091] and parameters: {'N_KNOTS': 9, 'HIDDEN_KEY': '128_64', 'DROPOUT': 0.09943242510363191, 'LR_P0': 0.0005371582324154472, 'LR_P1': 0.00011047117881763945, 'LR_P2': 0.0002965926016432513, 'LAMBDA_SMOOTH_P2': 0.00012227781489711028, 'LAMBDA_POS_P2': 0.4864764033332494, 'BATCH_SIZE': 32}.


trial=52 fold=2 seed=42 | R2=0.5074 MAE=0.4795 ElastScore=0.8999
Trial 52 summary | mean_R2=0.5958 std_R2=0.0993 robust_R2=0.5709 | mean_Elast=0.6026 std_Elast=0.2283 robust_Elast=0.5455

Trial 53
  N_KNOTS: 9
  HIDDEN_KEY: 128_64
  DROPOUT: 0.23558128080237772
  LR_P0: 0.0017986525170147936
  LR_P1: 0.00040621477715349933
  LR_P2: 0.0002965926016432513
  LAMBDA_SMOOTH_P2: 0.0005024169645947396
  LAMBDA_POS_P2: 0.3839938975560217
  BATCH_SIZE: 32
trial=53 fold=0 seed=11 | R2=0.7346 MAE=0.5097 ElastScore=0.8553
trial=53 fold=0 seed=29 | R2=0.7341 MAE=0.5093 ElastScore=0.9282
trial=53 fold=0 seed=42 | R2=0.7299 MAE=0.5129 ElastScore=0.7780
trial=53 fold=1 seed=11 | R2=0.6084 MAE=0.4973 ElastScore=0.8319
trial=53 fold=1 seed=29 | R2=0.4408 MAE=0.5054 ElastScore=0.4153
trial=53 fold=1 seed=42 | R2=0.6603 MAE=0.5033 ElastScore=0.5896
trial=53 fold=2 seed=11 | R2=0.4925 MAE=0.4891 ElastScore=0.6941
trial=53 fold=2 seed=29 | R2=0.4988 MAE=0.4822 ElastScore=0.7059


[I 2026-03-25 21:30:55,493] Trial 53 finished with values: [0.5666751505581351, 0.703433490146587] and parameters: {'N_KNOTS': 9, 'HIDDEN_KEY': '128_64', 'DROPOUT': 0.23558128080237772, 'LR_P0': 0.0017986525170147936, 'LR_P1': 0.00040621477715349933, 'LR_P2': 0.0002965926016432513, 'LAMBDA_SMOOTH_P2': 0.0005024169645947396, 'LAMBDA_POS_P2': 0.3839938975560217, 'BATCH_SIZE': 32}.


trial=53 fold=2 seed=42 | R2=0.4754 MAE=0.5006 ElastScore=0.9024
Trial 53 summary | mean_R2=0.5972 std_R2=0.1221 robust_R2=0.5667 | mean_Elast=0.7445 std_Elast=0.1644 robust_Elast=0.7034

Trials completados: 54
Trials Pareto-óptimos: 6


In [27]:
summary_rows = []
for t in study.trials:
    if t.values is None:
        continue
    row = {
        "trial": t.number,
        "mean_r2": t.user_attrs.get("mean_r2", np.nan),
        "std_r2": t.user_attrs.get("std_r2", np.nan),
        "mean_elast_score": t.user_attrs.get("mean_elast_score", np.nan),
        "std_elast_score": t.user_attrs.get("std_elast_score", np.nan),
        "mean_mae": t.user_attrs.get("mean_mae", np.nan),
        "mean_rmse": t.user_attrs.get("mean_rmse", np.nan),
        **t.params,
    }
    summary_rows.append(row)

df_trials_summary = pd.DataFrame(summary_rows).sort_values(
    ["mean_r2", "mean_elast_score"], ascending=[False, False]
)

print(df_trials_summary.head(15).to_string(index=False))

 trial  mean_r2   std_r2  mean_elast_score  std_elast_score  mean_mae  mean_rmse  N_KNOTS HIDDEN_KEY  DROPOUT    LR_P0    LR_P1    LR_P2  LAMBDA_SMOOTH_P2  LAMBDA_POS_P2  BATCH_SIZE
    41 0.642813 0.102166          0.616373         0.164099  0.491395   0.633481        3      64_32 0.166660 0.004896 0.000045 0.000665          0.000013       0.263288          64
    12 0.642596 0.094238          0.650589         0.145541  0.491504   0.635390       14     128_64 0.162260 0.001795 0.000049 0.000450          0.000030       0.458544          64
    47 0.639994 0.099568          0.696800         0.115148  0.494151   0.636840       13     128_64 0.105799 0.007054 0.000106 0.000242          0.000052       0.471834          32
    46 0.638757 0.102421          0.589355         0.156294  0.493666   0.637165       14  128_64_32 0.032025 0.008860 0.000722 0.000021          0.000024       0.330913          32
    26 0.637644 0.101385          0.524480         0.156640  0.492553   0.638141        6 

In [28]:
df_trials_summary["robust_score"] = (
    df_trials_summary["mean_r2"]
    - 0.25 * df_trials_summary["std_r2"].fillna(0.0)
    + 0.10 * df_trials_summary["mean_elast_score"]
)

best_row = df_trials_summary.sort_values("robust_score", ascending=False).iloc[0]

best_trial_payload = {
    "trial": int(best_row["trial"]),
    "robust_score": float(best_row["robust_score"]),
    "mean_r2": float(best_row["mean_r2"]),
    "std_r2": float(best_row["std_r2"]),
    "mean_elast_score": float(best_row["mean_elast_score"]),
    "std_elast_score": float(best_row["std_elast_score"]),
    "params": {
        "N_KNOTS": int(best_row["N_KNOTS"]),
        "HIDDEN_KEY": str(best_row["HIDDEN_KEY"]),
        "DROPOUT": float(best_row["DROPOUT"]),
        "LR_P0": float(best_row["LR_P0"]),
        "LR_P1": float(best_row["LR_P1"]),
        "LR_P2": float(best_row["LR_P2"]),
        "LAMBDA_SMOOTH_P2": float(best_row["LAMBDA_SMOOTH_P2"]),
        "LAMBDA_POS_P2": float(best_row["LAMBDA_POS_P2"]),
        "BATCH_SIZE": int(best_row["BATCH_SIZE"]),
    }
}

with open(BEST_TRIAL_PATH, "w", encoding="utf-8") as f:
    json.dump(best_trial_payload, f, indent=2, ensure_ascii=False)

df_trials_summary.to_csv(TRIAL_SUMMARY_PATH, index=False)

print("Best trial guardado en:", BEST_TRIAL_PATH)
print("Resumen trials guardado en:", TRIAL_SUMMARY_PATH)
print(json.dumps(best_trial_payload, indent=2, ensure_ascii=False))

Best trial guardado en: ../results/best_trial_params.json
Resumen trials guardado en: ../results/nn_hparam_trials_summary.csv
{
  "trial": 14,
  "robust_score": 0.6898868011601498,
  "mean_r2": 0.625558203160165,
  "std_r2": 0.11445097301152933,
  "mean_elast_score": 0.9294134125286713,
  "std_elast_score": 0.11389084110445409,
  "params": {
    "N_KNOTS": 10,
    "HIDDEN_KEY": "64_32_16",
    "DROPOUT": 0.17644785552418754,
    "LR_P0": 0.0006963825561558388,
    "LR_P1": 0.00011113650292557837,
    "LR_P2": 0.00020465153569806238,
    "LAMBDA_SMOOTH_P2": 0.01603732582410331,
    "LAMBDA_POS_P2": 0.2781456207286023,
    "BATCH_SIZE": 16
  }
}
